# The evidence model

A finding is a *statement about the data*; detectors supply *evidence* for
that statement. The separation is the backbone of the report: the public
taxonomy names what was discovered, while the algorithms that discovered it
live inside the evidence records. Two detectors that find the same thing
produce one finding with two evidence records — not two findings whose
names reveal which detector ran first.

This chapter is the methodology behind that design: how statements are
identified, how evidence is structured, how equivalent statements are
consolidated, and how the report selects what to show.

## One statement, several witnesses

The fixture below plants a single conditional regularity — premium
segments almost always ship express — and profiles it. Both rule detectors
(PRISM and the association miner) discover it, and the shared rule
evaluator re-measures it on a common partition:

In [ ]:
import json
import random

import polars as pl

import onceover

rng = random.Random(11)
n = 4000
segment = ["premium" if rng.random() < 0.5 else "std" for _ in range(n)]
shipping = [
    "express" if (rng.random() < 0.95 if s == "premium" else rng.random() < 0.05)
    else "ground"
    for s in segment
]
df = pl.DataFrame({"segment": segment, "shipping": shipping})

report = onceover.profile(df)
rule = next(f for f in report.findings if f["category"] == "conditional_rules")
print("claim:", rule["claim"])
print("conditions:", json.dumps(rule["specification"]["conditions"]))
print("outcome:", json.dumps(rule["specification"]["outcome"]))
print("detectors contributing evidence:", [e["method"]["detector"] for e in rule["evidence"]])

Three detectors, one statement. Each evidence record keeps its own
measurements — PRISM's score on uncovered positives, the association
miner's train/holdout counts, the evaluator's partition counts — and the
counts are never pooled or averaged. The claim the report shows comes from
the common evaluator, because its measurements are in the units every rule
shares: eligible rows, antecedent matches, joint matches, conditional rate,
baseline rate, and lift.

## The three dimensions of evidence

Every evidence record answers three questions separately. Conflating them
into one "confidence" label is what the schema is designed to avoid.

| Dimension | Values | What it answers |
|---|---|---|
| `basis` | `measured`, `estimated`, `inferred` | What kind of support is this — a count or comparison, a fitted quantity validated on held-out data, or an interpretation suggested by measurements? |
| `scope` | `sampling` (`full`/`sample`), `population_rows`, `rows_evaluated` | Which rows were actually covered, out of how many? |
| `validation` | `direct_comparison`, `cross_validation`, `diagnostic_holdout`, `diagnostic_partition`, `none` | How was the statement checked, and was the check independent? |

A `measured` basis with `full` scope and `direct_comparison` — a key's
uniqueness counted over every row, an identity's residuals scanned over
every pair-complete row — is the strongest evidence class the ledger
issues. An `estimated` finding carries the sample and holdout sizes so a
reader can judge it. An `inferred` role ("this column behaves like an
identifier") stays an interpretation even when its supporting counts cover
the full dataset: basis and scope are independent claims.

The `validation` field also records `used_for_selection`. A diagnostic
holdout that helped choose which model to report is not the same thing as
an independent confirmation, and the schema does not pretend otherwise.

## Consolidation

Two findings consolidate when they state the same thing. Equivalence is
decided by a canonical key: the category, the type, the subject (columns,
inputs, target), and the specification (form, conditions, outcome,
definitions) — after normalization. Rank, detector, and interestingness are
deliberately absent from the key, so a statement keeps its identity no
matter who discovered it or where it ranks.

When two evidence records arrive for one statement, the report keeps the
claim from the candidate with the broader evaluated coverage, appends all
evidence records, and fills in a missing interpretation if one arrives.
What it never does is merge the numbers: counts from overlapping samples or
different methods stay side by side, each labeled with its own scope.

Distinct statements stay distinct. `X → Y` and `Y → X` are different
claims. `X → Y` and `(X, Z) → Y` are different claims. "If X = a then Y = b"
differs from "X determines Y" the way a case differs from a rule: the
conditional statement covers one pocket of the data, the dependency covers
the mapping everywhere it was checked.

## Selection

The report selects before it renders. Selection ranks every distinct
statement by a deterministic score with three ingredients:

- **evidence quality** — direct full-scope comparison first, then
  cross-validated or holdout-validated estimates, then bare measured or
  inferred support;
- **effect strength**, in units native to each category — zero-violation
  dependencies score highest, rules are scored by improvement over
  baseline weighted by support, predictive dependencies by holdout R² or
  accuracy gain, associations by the strongest measured coefficient,
  concentration by the dominant value's share;
- **simplicity and specificity** — a stated form (a value mapping, an
  affine formula, a piecewise model) outranks an unspecified one; more
  inputs or conditions discount slightly.

Report-wide limits apply across detectors: at most 100 findings, 20
conditional rules, 2 rules per outcome, and 3 predictive findings per
target. The `selection` block records how many candidates entered, how
many survived verification, how many distinct statements they formed, and
how many were selected — so a reader can tell consolidation from
suppression.

In [ ]:
print(json.dumps(report.data["selection"], indent=2))
from collections import Counter
print("selected by category:", dict(Counter(f["category"] for f in report.findings)))

## What evidence is not

The ledger issues no p-values and no multiple-testing guarantees. A
conditional rate measured on the diagnostic partition is exactly that — a
rate on disclosed rows — and the partition may have been seen during
discovery. An absence is not a negation: a finding can be missing because
the property is absent, because the pass was capped, or because it was
never run. The coverage block distinguishes those cases, and reading it is
part of reading the report ([](../usage/coverage.ipynb)).

The field-level reference for every structure in this chapter is the
[report contract](../appendix/report_contract.ipynb); the schema reference is
[docs/findings.md](https://github.com/fcela/onceover/blob/main/docs/findings.md).